<small><a href="https://raw.githack.com/nami-rangana/mdCIF/v1/examples/view/python/view_mdcif.html" target="_blank">Click here</a> to view as a webpage (some widgets may be missing).</small>

# Viewing an mdCIF file in Python with py3Dmol

An mdCIF file captures a simulation’s main states as model structures, plus three major modes for each state. A mode shows how the atoms move corelated motions. In this notebook, we explore how we can open a mdCIF file and visualize the modes as animations or as porcupine plots, where arrows show the motions of Cα atoms.

We’ll use the GB1 example from [`1_basic.ipynb`](../../create/1_basic/1_basic.ipynb), which represents a 300 K MELD simulation with two states.

**You’ll need:** the `mdcif` package (installed with this repository, see its README) and [py3Dmol](https://pypi.org/project/py3Dmol/). The package’s reader, `mdcif.reader.read_mdcif`, opens text mmCIF and BinaryCIF (`.bcif`) files alike, and its viewer, `mdcif.view.MdcifViewer`, draws what they hold with py3Dmol.

In [1]:
import os
import sys

os.environ.setdefault("OMP_NUM_THREADS", "4")  # 4 OpenMp threads
sys.path = [p for p in sys.path if "site-packages" not in p or p.startswith(sys.prefix)]

In [2]:
from pathlib import Path

import ihm
import numpy as np
import py3Dmol

import mdcif
from mdcif.cli import info
from mdcif.reader import read_mdcif
from mdcif.view import MdcifViewer, save_png

print(f"mdcif {mdcif.__version__} (python-ihm {ihm.__version__})")

CIF = Path("../../create/1_basic/gb1_meld_300K.cif")  # your own file works the same way (or .bcif)
FIGS = Path("figures")                                # saved pictures go here (PNG)
if not CIF.exists():
    raise FileNotFoundError(f"{CIF}: build it first with ../../create/1_basic/1_basic.ipynb "
                            "(or `mdcif build`, see ../../create/README.md)")
FIGS.mkdir(exist_ok=True)

mdcif 0.0.1 (python-ihm 2.11)


## 1. Open the file

`read_mdcif` reads the whole file into one object, `data`:

| Attribute | What it holds |
|---|---|
| `data.system` | the structures, the states and the PDB entry’s details, as python-ihm objects |
| `data.simulation` | the simulation conditions |
| `data.mode_sets` | each state’s own modes, by model number |
| `data.projection` | every frame’s position on the global PCs, and its state |
| `data.global_modes` | the modes of the whole run, saved to visualize the clusters |

`info` prints the same summary as `mdcif info` on the command line.

In [3]:
data = read_mdcif(CIF)
info(CIF)

../../create/1_basic/gb1_meld_300K.cif: Protein G B1 domain (PDB 3GB1) folded with MELD x Amber: 300 K replica ensemble


  simulation: force_field=ff19SB, water_model=implicit: GB-Neck2 (igb=8, mbondi3 radii), ensemble=NVT, temperature_K=300.0, timestep_fs=4.0, length_ns=403.65, n_replicas=30, save_interval_ps=2.0, enhanced_sampling=MELD: temperature + Hamiltonian replica exchange


  global: 3 PCA modes of 17683 frames on 56 atoms, var% 60/11/8


  projections: 5000 frames on global modes 1-3


  model 1: Cluster 1 medoid (frame 142700, t = 285.402 ns), 435 atoms, 16847 frames, precision 1.61 A, 3 PCA modes on 56 atoms, var% 26/20/8


      16847 of 17683 frames (95.3%); medoid CA RMSD to PDB 3GB1 model 1 2.61 A


  model 2: Cluster 2 medoid (frame 127120, t = 254.242 ns), 435 atoms, 836 frames, precision 0.93 A, 3 PCA modes on 56 atoms, var% 20/14/9


      836 of 17683 frames (4.7%); medoid CA RMSD to PDB 3GB1 model 1 9.04 A


In [4]:
data.simulation

{'force_field': 'ff19SB',
 'water_model': 'implicit: GB-Neck2 (igb=8, mbondi3 radii)',
 'ensemble': 'NVT',
 'temperature_K': 300.0,
 'timestep_fs': 4.0,
 'length_ns': 403.65,
 'n_replicas': 30,
 'save_interval_ps': 2.0,
 'enhanced_sampling': 'MELD: temperature + Hamiltonian replica exchange',
 'details': 'MELD (Modeling Employing Limited Data) folding from an extended chain built from sequence with tleap. 30 replicas, geometric temperature ladder 300-550 K over the first 30% of replicas; exchanges every 50 ps (12500 steps). Restraints: secondary structure from ss.dat (28 helix/strand windows, 85% active, constant strength), strand pairing (213 groups, 10 active) and hydrophobic contacts (121 groups, 21 active), both scaled nonlinearly from 1.0 to 0.4 along the replica ladder and faded in over the first 200 exchanges. Langevin thermostat (gamma 2/ps), SHAKE on bonds to hydrogen, no nonbonded cutoff. Analysed: replica 1 (300 K, full restraint strength; Amber file meld.nc.001).'}

----

Now lets see how many models are available in our loaded file using `gemmi`:

In [5]:
import gemmi
print(f'number of models: {len(gemmi.read_structure(str(CIF)))}')

number of models: 2


Before visualizing the models and their modes, we can dig more information about these model using a code block similar to following. This gives you results from PCA:

In [6]:
def sigma(ms): #std
    return np.sqrt(ms.cartesian_variance)


sets = {f"state {m}": ms for m, ms in data.mode_sets.items()} | {"global": data.global_modes}
for name, ms in sets.items():
    unit = "amu Å²" if ms.mass_weighted else "Å²"
    print(f"{name}: {ms.method} of {ms.n_frames} frames, on the atoms of model {ms.model_id}")
    for k, (ev, s, share) in enumerate(zip(ms.eigenvalues, sigma(ms), ms.variance_fraction,
                                           strict=True), 1):
        print(f"  mode {k}: eigenvalue {ev:5.1f} {unit}, σ = {s:4.1f} Å, "
              f"{100 * share:2.0f}% of the motion")

ms = data.mode_sets[1]
print("\nstate 1 eigenvectors:", ms.eigenvectors.shape, "= modes \u00D7 atoms \u00D7 (dx, dy, dz)")

state 1: PCA of 16847 frames, on the atoms of model 1
  mode 1: eigenvalue  37.4 Å², σ =  6.1 Å, 26% of the motion
  mode 2: eigenvalue  28.5 Å², σ =  5.3 Å, 20% of the motion
  mode 3: eigenvalue  10.8 Å², σ =  3.3 Å,  8% of the motion
state 2: PCA of 836 frames, on the atoms of model 2
  mode 1: eigenvalue   7.6 Å², σ =  2.8 Å, 20% of the motion
  mode 2: eigenvalue   5.6 Å², σ =  2.4 Å, 14% of the motion
  mode 3: eigenvalue   3.3 Å², σ =  1.8 Å,  9% of the motion
global: PCA of 17683 frames, on the atoms of model 1
  mode 1: eigenvalue 203.1 Å², σ = 14.3 Å, 60% of the motion
  mode 2: eigenvalue  35.5 Å², σ =  6.0 Å, 11% of the motion
  mode 3: eigenvalue  25.6 Å², σ =  5.1 Å,  8% of the motion

state 1 eigenvectors: (3, 56, 3) = modes × atoms × (dx, dy, dz)


**The viewer.** `MdcifViewer` takes the file we just read, collects each model’s atoms and coordinates once, and draws them with py3Dmol:

| Method | What it shows |
|---|---|
| `viewer.show()` | all models together, one colour each; `viewer.show(2)` shows only model 2 |
| `viewer.porcupine(model)` | a porcupine plot of each of the model’s modes |
| `viewer.movie(model)` | the model moving along each of its modes |
| `save_png(view, path)` | saves any view as a PNG picture |

Each drawing method gives back an ordinary py3Dmol view: `.show()` displays it, and py3Dmol’s own methods (`rotate`, `zoom`, ...) work on it too. Add `mode=1` to draw a single mode, or `scope="global"` to draw the modes of the whole run instead of the state’s own. The mode set’s `atom_indices` tell us which atoms’ movements are recorded, here the 56 Cα atoms.

In [7]:
viewer = MdcifViewer(data)                       # MdcifViewer(CIF) would read the file itself
for ens in data.system.ensembles:                # one IHM ensemble per state
    model = ens.model_group[0]                   # its one model, the medoid
    m = int(model._id)                           # the model's number in the file
    print(f"model {m}: {len(viewer.atoms(m))} atoms, modes on "
          f"{len(data.mode_sets[m].atom_indices)} of them, "
          f"{100 * ens.num_models / data.global_modes.n_frames:.0f}% of the frames; {model.name}")
viewer

model 1: 435 atoms, modes on 56 of them, 95% of the frames; Cluster 1 medoid (frame 142700, t = 285.402 ns)
model 2: 435 atoms, modes on 56 of them, 5% of the frames; Cluster 2 medoid (frame 127120, t = 254.242 ns)


<MdcifViewer: models [1, 2], own modes on models [1, 2], 3 global modes>

## 2. Visualize the models

Both models are aligned to the NMR structure 3GB1, so we can view them together and compare their shapes. The labels in the corner give each model’s colour. To look at one model on its own, give its number, e.g. `viewer.show(2)`; `viewer.show(1, 2)` picks several.

In [8]:
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3. Porcupine plots

#### 3.1. Vizualize:
A porcupine plot turns a mode into a snapshot, with arrows showing how far each Cα moves at 3σ (argument `scale`). Tiny movements under 0.5 Å (argument `min_length`) are left out. Modes move in both directions, so the arrows could point the other way too. The panels turn together: drag any of them.

In [9]:
viewer.porcupine(1).show()                       # first model, its three modes
viewer.porcupine(2).show()                       # second model

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### 3.2. Save visualization as image

Now we'll save a view as a `png` image file. py3Dmol draws in the browser, so Python never sees the picture: `save_png` opens the view as a web page in a headless Chrome and saves a screenshot. It finds Chrome (or Chromium, or Edge) by itself; if it can't, set the `CHROME` environment variable to the program, or pass `chrome=`. 3Dmol.js is loaded from the internet.

Here we save the first mode of the first model, rotated and zoomed, as a PNG 6 inches wide (600 dpi):

In [10]:
view = viewer.porcupine(1, mode=1)               # one panel: mode 1 of model 1
view.rotate(20, "y").rotate(30, "x").rotate(15, "z").zoom(1.5)
save_png(view, FIGS / "porcupine_1_mode_1.png", width_in=6.0)

PosixPath('figures/porcupine_1_mode_1.png')

Here's how it looks:
<p align="center">
  <img src="figures/porcupine_1_mode_1.png" alt="Project Screenshot" style="width: 100%; max-width: 100%; height: auto;">
</p>

## 4. Bringing the Three Modes to Life

Each mode gives every Cα a direction and a typical displacement, measured in Å. `viewer.movie` turns that information into an animation, moving the structure back and forth up to 3σ (argument `amplitude`). Other backbone atoms follow their residue’s Cα.

These modes follow straight paths, so large motions can stretch bonds or loops. That’s a limitation of the approximation, not the file. Real protein motions follow more complex paths.

In [11]:
viewer.movie(1, n_frames=25).show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

**Movie 1.** Watch the folded state move along modes 1, 2, and 3, each up to 3σ. Drag any panel to rotate all three together. Labels show each mode’s σ and share of the state’s motion.

Run `viewer.movie(2)` to explore the other state:

In [12]:
viewer.movie(2, n_frames=50).show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

**Which residues move most?** The movies and porcupine plots show the motion. Here, we measure it: for each mode, the cell lists the six Cα atoms with the largest 3σ displacements, alongside the median for all 56 atoms. To explore the misfolded state, change the model number from 1 to 2 in the first two lines.

In [13]:
atoms = viewer.atoms(1)
ms = data.mode_sets[1]
on = [atoms[i] for i in ms.atom_indices]          # the Cα the modes are stored on
for k in range(1, ms.n_modes + 1):
    length = 3 * sigma(ms)[k - 1] * np.linalg.norm(ms.eigenvectors[k - 1], axis=1)   # Å, as drawn
    top = np.argsort(length)[::-1][:6]
    print(f"mode {k}: " + ", ".join(f"{on[i].resname.title()}{on[i].resnum} {length[i]:.1f}"
                                    for i in top)
          + f" Å  (median over all Cα: {np.median(length):.1f} Å)")

mode 1: Leu12 7.8, Lys13 6.5, Thr11 6.3, Val39 4.9, Asp36 4.6, Asn37 4.6 Å  (median over all Cα: 1.2 Å)
mode 2: Leu12 5.6, Val39 5.4, Glu56 5.0, Gly38 4.5, Thr11 4.4, Lys13 4.3 Å  (median over all Cα: 1.1 Å)
mode 3: Leu12 3.9, Thr11 3.4, Gly9 3.0, Lys10 2.9, Lys13 2.5, Ala20 1.8 Å  (median over all Cα: 0.8 Å)


In the folded state the motion sits in the loops, while the strands and the helix hardly move
(median arrow about 1 Å). The β1–β2 turn (residues 9–13) has the longest arrows in all three
modes. In mode 1 it moves together with the loop after the helix (residues 36–39); in mode 2
with that loop and the last residue (Glu56); mode 3 is almost only the β1–β2 turn.

## 5. More with py3Dmol: pick residues, change the look

Every picture in this notebook is an ordinary py3Dmol view, so py3Dmol’s own tools work on it before you call `.show()`. Two ideas cover most of them:

- A **selection** picks atoms, written as a dictionary: `{"resi": "9-13"}` is residues 9 to 13, `{"resi": [12, 39]}` residues 12 and 39, `{"resn": "LEU"}` every leucine, `{"atom": "CA"}` the Cα atoms. Residue numbers are the ones in the file, as in the table above.
- A **style** says how to draw them: `cartoon`, `stick`, `sphere` or `line`, each with options such as `color`, `radius` or `opacity`.

`view.setStyle(selection, style)` replaces how the selected atoms are drawn, `view.addStyle(selection, style)` draws them a second way on top, and `view.zoomTo(selection)` points the camera at them.

#### 5.1. Highlight residues

The β1–β2 turn (residues 9–13) moves the most in all three modes of the folded state. Here it is in model 1, with its side chains as sticks and every residue labelled:

In [14]:
turn = {"resi": "9-13"}                          # the β1–β2 turn

view = viewer.models(1)
view.addStyle(turn, {"stick": {"radius": 0.2}})  # side chains on top of the cartoon
view.addResLabels(turn, {"fontSize": 11, "fontColor": "black", "backgroundColor": "white",
                         "backgroundOpacity": 0.7})
view.zoomTo(turn)
view.zoom(0.6)                                   # step back a little, to keep some context
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### 5.2. Show only some residues

To draw only part of the protein, first hide everything with an empty style, then draw the residues you want. py3Dmol numbers the models in a view 0, 1, ... in the order they were drawn, so `"model": i` gives each model its own colour back. Here are residues 1–20 (β1, the turn and β2): the hairpin of the folded model 1 (blue) next to the same residues in the misfolded model 2 (orange).

In [15]:
hairpin = {"resi": "1-20"}                       # β1, the turn and β2

view = viewer.models()                           # both models
view.setStyle({}, {})                            # hide everything...
for i, m in enumerate(viewer.model_ids):         # ...then draw residues 1-20 of each model
    view.setStyle({"model": i, **hairpin}, {"cartoon": {"color": viewer.color(m)}})
view.zoomTo(hairpin)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### 5.3. Residues in porcupine plots and movies

The grids of sections 3 and 4 work the same way: a style without `viewer=` applies to every panel, and `viewer=(0, 1)` would pick the second panel only. Here the turn is coloured in all three porcupine panels, which shows that it carries the longest arrows. Hiding residues works less well in porcupine plots: the arrows are not atoms, so they stay where they are.

In [16]:
view = viewer.porcupine(1)
view.setStyle(turn, {"cartoon": {"color": "#b8336a"}})   # the turn, in every panel
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### 5.4. Visual effects

A few more of py3Dmol’s options: a rainbow along the chain (`"spectrum"`, from the N-terminus in purple to the C-terminus in red), β-strands drawn as arrows, a see-through molecular surface, an outline and a background colour. `spin` makes the view turn by itself: remove the `#` to try it.

In [17]:
view = viewer.models(1, legend=False)
view.setStyle({"cartoon": {"color": "spectrum", "arrows": True}})
view.addSurface(py3Dmol.SES, {"opacity": 0.25, "color": "white"})   # or VDW, SAS, MS
view.setViewStyle({"style": "outline", "color": "black", "width": 0.03})
view.setBackgroundColor("#f4f4f2")
# view.spin("y", 0.5)                            # turn slowly about the y axis
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

**See more.** py3Dmol passes each call on to the JavaScript library 3Dmol.js, so the 3Dmol.js documentation lists everything you can use; in Python, the same arguments are written as dictionaries.

- [Selections](https://3dmol.org/doc/AtomSelectionSpec.html): every key a selection can use (`resi`, `resn`, `chain`, `within`, `byres`, ...)
- [Styles](https://3dmol.org/doc/AtomStyleSpec.html), [cartoon options](https://3dmol.org/doc/CartoonStyleSpec.html) and [labels](https://3dmol.org/doc/LabelSpec.html)
- [Viewer methods](https://3dmol.org/doc/GLViewer.html): `setStyle`, `addStyle`, `addSurface`, `addResLabels`, `setViewStyle`, `spin`, `zoomTo`, ...
- The [py3Dmol example notebook](https://github.com/3dmol/3Dmol.js/blob/master/py3Dmol/examples.ipynb) and the [3Dmol.js tutorials](https://3dmol.org/doc/tutorial-home.html)